<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/Workflow_Patterns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workflow Patterns: Chaining, Parallelization, Routing, and Orchestrator-Workers

Between "one prompt, one answer" and "an agent that decides for itself" sits a large, useful middle ground: **workflows**, where you decide the steps in code and the model fills each one in. Most systems that ship are workflows. This notebook builds the four patterns you will reach for most often, on one running example: turning three documentation pages into a grounded FAQ page.

## What You'll Learn

- Where a single do-everything call stops being the right shape, measured rather than assumed
- **Chaining**: one job per call, with programmatic gates between the steps
- **Parallelization**: `asyncio.gather` over independent steps, with a concurrency limit that keeps you inside rate limits
- **Routing**: a typed classification call plus plain `if`/`elif` dispatch
- **Orchestrator-workers**: an LLM that decides which typed sub-tasks to run, workers that run them, and a synthesizer that writes the reply
- What each pattern costs, so you can tell when it is worth it

## 1. Setup

The standard course setup cell: pick a provider, and it installs the pinned dependencies (in Colab) and loads the matching API key from Colab Secrets or from a `.env` file at the repo root. We default to Gemini because its free tier covers this notebook. Switch `PROVIDER` to use another.

In [1]:
# ============================================================
# Setup - environment, dependencies, API keys, provider
# ============================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules

# Pick your model provider (dropdown in Colab; edit the value locally)
PROVIDER = "gemini"  # @param ["gemini", "openai", "anthropic"]

# Pick a model for the selected provider - or TYPE any newer model ID into the
# box (the dropdown is editable thanks to allow-input):
CHAT_MODEL = "gemini-3.5-flash-lite"  # @param ["gemini-3.5-flash-lite", "gpt-5.6-luna", "claude-haiku-4-5"] {allow-input: true}

REQUIRED_KEYS = {
    "gemini": ["GOOGLE_API_KEY"],
    "openai": ["OPENAI_API_KEY"],
    "anthropic": ["ANTHROPIC_API_KEY"],
}[PROVIDER]  # only the selected provider's key is required

if IN_COLAB:
    import importlib
    import site
    import subprocess

    # Shared install profile, pinned course-wide (pin set checked August 13, 2026).
    # Library updates can change behavior, so we pin.
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "google-genai==2.18.0",
            "openai==3.0.0",
            "anthropic==0.122.0",
        ],
        check=True,
    )
    importlib.reload(site)  # make new packages importable without a runtime restart

    # In Colab: Secrets tab (key icon in the left sidebar) -> Add new secret ->
    # name it e.g. GOOGLE_API_KEY, paste the key, and toggle notebook access on.
    from google.colab import userdata

    for key in REQUIRED_KEYS:
        os.environ[key] = userdata.get(key)

if not IN_COLAB:
    # Locally: API keys live in a .env file at the repo root.
    from dotenv import load_dotenv

    load_dotenv()
    missing = [k for k in REQUIRED_KEYS if not os.getenv(k)]
    assert not missing, f"Missing from .env: {missing}"

print(f"Setup complete - {'Colab' if IN_COLAB else 'local'} | provider: {PROVIDER}")

Setup complete - local | provider: gemini


## 2. Two Helpers: `generate()` and `generate_typed()`

Every pattern below is built from two calls. `generate()` returns text, and it is the helper from the "How To Use LLMs via API" notebook. `generate_typed()` is its structured-output sibling from Section 6: you hand it a Pydantic model and get an instance back instead of a string you have to parse.

`generate_typed()` is what makes workflows composable. A step whose output is a validated object can be inspected, filtered, and branched on in ordinary Python, which is exactly what the steps below do to each other.

In [2]:
from anthropic import Anthropic
from google import genai
from google.genai import types as genai_types
from openai import OpenAI

# Course-standard default models per provider (checked August 2026)
MODELS = {
    "gemini": "gemini-3.5-flash-lite",
    "openai": "gpt-5.6-luna",
    "anthropic": "claude-haiku-4-5",
}
MODELS[PROVIDER] = CHAT_MODEL  # the setup-cell selection wins

if PROVIDER == "gemini":
    gemini_client = genai.Client()
elif PROVIDER == "openai":
    openai_client = OpenAI()
elif PROVIDER == "anthropic":
    anthropic_client = Anthropic()

# Every call is counted here, so we can price the whole notebook at the end.
USAGE = {"calls": 0, "input_tokens": 0, "output_tokens": 0}


def _record(input_tokens, output_tokens):
    USAGE["calls"] += 1
    USAGE["input_tokens"] += input_tokens or 0
    USAGE["output_tokens"] += output_tokens or 0


def generate(prompt, system=None, model=None):
    """Send one prompt to the selected PROVIDER and return the reply text."""
    if PROVIDER == "gemini":
        response = gemini_client.models.generate_content(
            model=model or MODELS["gemini"],
            contents=prompt,
            config=genai_types.GenerateContentConfig(system_instruction=system),
        )
        usage = response.usage_metadata
        _record(usage.prompt_token_count, usage.candidates_token_count)
        return response.text

    if PROVIDER == "openai":
        response = openai_client.responses.create(
            model=model or MODELS["openai"],
            instructions=system,
            input=prompt,
            reasoning={"effort": "none"},
        )
        _record(response.usage.input_tokens, response.usage.output_tokens)
        return response.output_text

    if PROVIDER == "anthropic":
        response = anthropic_client.messages.create(
            model=model or MODELS["anthropic"],
            max_tokens=4096,
            # Anthropic rejects system=None, so only pass it when set
            **({"system": system} if system else {}),
            messages=[{"role": "user", "content": prompt}],
        )
        _record(response.usage.input_tokens, response.usage.output_tokens)
        return response.content[0].text

    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


The typed version takes a Pydantic class and returns an instance of it. Gemini and OpenAI accept the schema directly. Anthropic has no separate structured-output parameter here, so we define the schema as a single tool and force the model to call it, which is the long-standing way to get guaranteed-shape JSON out of the Messages API.

In [3]:
def generate_typed(prompt, schema, system=None, model=None):
    """Same call, but the reply comes back parsed into the Pydantic `schema`."""
    if PROVIDER == "gemini":
        response = gemini_client.models.generate_content(
            model=model or MODELS["gemini"],
            contents=prompt,
            config=genai_types.GenerateContentConfig(
                system_instruction=system,
                response_mime_type="application/json",
                response_schema=schema,
            ),
        )
        usage = response.usage_metadata
        _record(usage.prompt_token_count, usage.candidates_token_count)
        return response.parsed

    if PROVIDER == "openai":
        response = openai_client.responses.parse(
            model=model or MODELS["openai"],
            instructions=system,
            input=prompt,
            text_format=schema,
            reasoning={"effort": "none"},
        )
        _record(response.usage.input_tokens, response.usage.output_tokens)
        return response.output_parsed

    if PROVIDER == "anthropic":
        response = anthropic_client.messages.create(
            model=model or MODELS["anthropic"],
            max_tokens=4096,
            **({"system": system} if system else {}),
            messages=[{"role": "user", "content": prompt}],
            tools=[{
                "name": "record",
                "description": f"Record the result as {schema.__name__}.",
                "input_schema": schema.model_json_schema(),
            }],
            tool_choice={"type": "tool", "name": "record"},
        )
        _record(response.usage.input_tokens, response.usage.output_tokens)
        return schema.model_validate(response.content[0].input)

    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


def show(title, body):
    """Print a labelled block, so long outputs stay readable."""
    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")
    print(body)

## 3. The Task and Its Source Material

The running example is a job every documentation team has: turn a set of pages into an FAQ that answers real reader questions and cites the page each answer came from. Three short pages are enough to show every pattern, and they keep the token bill near zero.

*Model outputs in this notebook were captured in August 2026 on `gemini-3.5-flash-lite`. LLMs are non-deterministic and they get replaced, so **your output will differ in wording**. What should stay stable is the shape of each result and the comparison each cell is making.*

In [4]:
page_chunking = {
    "title": "Chunking Documents for Retrieval",
    "content": """
    Retrieval systems do not search whole documents, they search pieces of them.
    Splitting a document into pieces is called chunking, and the size of those
    pieces decides what the retriever can find. Chunks that are too small lose the
    context that makes a passage meaningful. Chunks that are too large dilute the
    passage with unrelated text, so the match score drops. A practical default is
    to split on headings first, so each chunk covers one logical unit, then cap
    the result at a fixed token count with a small overlap between neighbours so
    that an idea cut in half still appears whole in one of the two chunks. Code
    blocks and tables should never be split, because half a code block is not
    runnable and half a table has no header row.
    """,
}

page_embeddings = {
    "title": "Embeddings and Vector Search",
    "content": """
    An embedding model turns a piece of text into a fixed-length vector of
    numbers. Texts with similar meaning land close together in that vector space,
    which is what lets a retriever match a question to a passage that answers it
    without sharing any of its words. At query time the question is embedded with
    the same model used at ingestion time, and the store returns the chunks whose
    vectors are nearest, usually by cosine similarity. Using a different model for
    queries than for documents produces vectors in two unrelated spaces, and the
    scores become meaningless. Embeddings are weak at exact identifiers such as
    error codes or function names, which is why many systems run keyword search
    alongside them and merge the two result lists.
    """,
}

page_reranking = {
    "title": "Reranking Retrieved Chunks",
    "content": """
    A first-stage retriever is tuned for speed, so it compares a query vector
    against millions of document vectors that were computed before the query
    existed. A reranker does the opposite: it reads the query and one candidate
    chunk together and scores how well that chunk answers that query. Because it
    scores every pair on demand, it is far more accurate and far more expensive,
    so it runs over a shortlist rather than the whole corpus. The usual shape is
    to retrieve fifty to a hundred candidates cheaply, rerank them, and keep the
    top handful for the prompt. The cost is latency: one extra network call
    between retrieval and generation, on every single query.
    """,
}

SOURCES = [page_chunking, page_embeddings, page_reranking]
SOURCE_TITLES = [source["title"] for source in SOURCES]

# One blob holding all three pages, for the steps that need to see everything.
combined_content = "\n\n".join(
    f"Source Title: {s['title']}\nContent: {s['content'].strip()}" for s in SOURCES
)

print(SOURCE_TITLES)
print(f"{len(combined_content.split())} words of source material")

['Chunking Documents for Retrieval', 'Embeddings and Vector Search', 'Reranking Retrieved Chunks']
389 words of source material


## 4. Attempt One: A Single Call That Does Everything

Start with the version everyone writes first. One prompt asks the model to invent the questions, answer them, keep each answer short, cite the pages it used, and balance the coverage across all three pages. Four rules, one call.

We also define the output schema up front. `FAQ` and `FAQList` are the Pydantic models from Section 6, and they are reused by every later version of this pipeline, which is the point: the schema is the contract, and the patterns below only change how the contract gets filled.

In [5]:
from pydantic import BaseModel, Field

N_PER_SOURCE = 4
N_QUESTIONS = N_PER_SOURCE * len(SOURCES)  # 12
MAX_ANSWER_WORDS = 10


class FAQ(BaseModel):
    """One question, its answer, and the source titles the answer came from."""

    question: str = Field(description="The question a reader might ask")
    answer: str = Field(description="The answer, at most 10 words")
    sources: list[str] = Field(description="Exact source titles used for the answer")


class FAQList(BaseModel):
    faqs: list[FAQ] = Field(description="The generated FAQ entries")


prompt_everything = """
You are writing an FAQ page for a documentation site.

From the provided content, produce exactly {n} frequently asked questions.
All four rules matter:
1. Return exactly {n} entries.
2. Every answer is at most {max_words} words and uses ONLY the provided content.
3. Every answer lists the exact 'Source Title' values it drew on, and no others.
4. Each of the three sources is the first listed source of at least {per_source}
   answers, so the page covers all three pages evenly.

<provided_content>
{content}
</provided_content>
""".strip()


def one_big_call():
    result = generate_typed(
        prompt_everything.format(
            n=N_QUESTIONS,
            max_words=MAX_ANSWER_WORDS,
            per_source=N_PER_SOURCE,
            content=combined_content,
        ),
        FAQList,
    )
    return result.faqs


faqs_single = one_big_call()
show("First three entries from the single call",
     "\n".join(faq.model_dump_json(indent=2) for faq in faqs_single[:3]))


First three entries from the single call
{
  "question": "What is splitting a document called?",
  "answer": "It is called chunking.",
  "sources": [
    "Chunking Documents for Retrieval"
  ]
}
{
  "question": "Why do chunks that are too small fail?",
  "answer": "They lose the context that makes a passage meaningful.",
  "sources": [
    "Chunking Documents for Retrieval"
  ]
}
{
  "question": "What happens when chunks are too large?",
  "answer": "The match score drops due to unrelated text.",
  "sources": [
    "Chunking Documents for Retrieval"
  ]
}


Now grade it. Three of the four rules are mechanically checkable, so we check them in code rather than reading the output and hoping. Note that `check_faqs` never looks at how the list was produced, which lets us reuse it unchanged on every later version.

In [6]:
from collections import Counter


def check_faqs(faqs):
    """Score an FAQ list against the four rules the prompt asked for."""
    primary = Counter(faq.sources[0] for faq in faqs if faq.sources)
    per_source = {title: primary.get(title, 0) for title in SOURCE_TITLES}
    report = {
        "returned": len(faqs),
        "count_ok": len(faqs) == N_QUESTIONS,
        "over_word_limit": sum(
            1 for faq in faqs if len(faq.answer.split()) > MAX_ANSWER_WORDS
        ),
        "invented_sources": sum(
            1 for faq in faqs for s in faq.sources if s not in SOURCE_TITLES
        ),
        "per_source": per_source,
    }
    report["all_rules_ok"] = (
        report["count_ok"]
        and report["over_word_limit"] == 0
        and report["invented_sources"] == 0
        and min(per_source.values()) >= N_PER_SOURCE
    )
    return report


passed = 0
for run in range(1, 6):
    report = check_faqs(one_big_call())
    passed += report["all_rules_ok"]
    print(f"run {run}: {report}")

print(f"\n{passed} of 5 runs satisfied all four rules")

run 1: {'returned': 12, 'count_ok': True, 'over_word_limit': 0, 'invented_sources': 0, 'per_source': {'Chunking Documents for Retrieval': 4, 'Embeddings and Vector Search': 4, 'Reranking Retrieved Chunks': 4}, 'all_rules_ok': True}


run 2: {'returned': 12, 'count_ok': True, 'over_word_limit': 0, 'invented_sources': 0, 'per_source': {'Chunking Documents for Retrieval': 4, 'Embeddings and Vector Search': 4, 'Reranking Retrieved Chunks': 4}, 'all_rules_ok': True}


run 3: {'returned': 12, 'count_ok': True, 'over_word_limit': 0, 'invented_sources': 0, 'per_source': {'Chunking Documents for Retrieval': 4, 'Embeddings and Vector Search': 4, 'Reranking Retrieved Chunks': 4}, 'all_rules_ok': True}


run 4: {'returned': 12, 'count_ok': True, 'over_word_limit': 0, 'invented_sources': 0, 'per_source': {'Chunking Documents for Retrieval': 4, 'Embeddings and Vector Search': 4, 'Reranking Retrieved Chunks': 4}, 'all_rules_ok': True}


run 5: {'returned': 12, 'count_ok': True, 'over_word_limit': 0, 'invented_sources': 0, 'per_source': {'Chunking Documents for Retrieval': 4, 'Embeddings and Vector Search': 4, 'Reranking Retrieved Chunks': 4}, 'all_rules_ok': True}

5 of 5 runs satisfied all four rules


If you expected a mess, that is the first thing worth updating. On a task this size a current model follows all four rules most of the time, and the single call is the cheapest and fastest version we will build all notebook. Splitting it up is not something you do because the model is too weak.

You split it up when you need to do something with the middle. Here is the cheapest way to see the difference: suppose a reviewer rejects one entry out of twelve. Measure what each design costs to fix it.

In [7]:
def tokens_for(fn, *args, **kwargs):
    """Run fn and report only the tokens that call consumed."""
    before = dict(USAGE)
    fn(*args, **kwargs)
    return {
        "calls": USAGE["calls"] - before["calls"],
        "input_tokens": USAGE["input_tokens"] - before["input_tokens"],
        "output_tokens": USAGE["output_tokens"] - before["output_tokens"],
    }


rejected = faqs_single[5]
show("The entry a reviewer rejected", rejected.model_dump_json(indent=2))

# Repair option A: the single call has one unit of work, so we regenerate all 12.
print("regenerate the whole page:", tokens_for(one_big_call))

# Repair option B: regenerate just this answer, which is what a chain lets us do.
one_answer_prompt = """
Answer the question in at most {max_words} words, using ONLY the provided content.

<question>
{question}
</question>

<provided_content>
{content}
</provided_content>
""".strip().format(
    max_words=MAX_ANSWER_WORDS,
    question=rejected.question,
    content=combined_content,
)

print("regenerate one answer:  ", tokens_for(generate, one_answer_prompt))


The entry a reviewer rejected
{
  "question": "Why do similar texts land close together?",
  "answer": "To let a retriever match questions to answering passages.",
  "sources": [
    "Embeddings and Vector Search"
  ]
}


regenerate the whole page: {'calls': 1, 'input_tokens': 633, 'output_tokens': 647}


regenerate one answer:   {'calls': 1, 'input_tokens': 554, 'output_tokens': 11}


Look at the output side, which is the expensive side. Fixing one entry through the single call means paying to regenerate all twelve, because the call is the unit of work. The targeted repair pays for one sentence. The input side barely moves here only because both prompts carry the same three pages.

That gap is the argument for chaining, and it applies to every intervention, not only repairs: a review step, a cache, a cheaper model for the mechanical steps, a validator that runs before anything reaches a reader. A single call has no seams to put those things in.

## 5. Chaining: One Job Per Call, With Gates In Between

**Prompt chaining** decomposes a task into a fixed sequence of calls, where each call's output feeds the next [(Anthropic, 2024)](https://www.anthropic.com/engineering/building-effective-agents). The sequence is fixed by us, in code. The model never chooses what happens next.

Our chain has three steps: write questions for one page, answer one question, attribute one answer. Between the steps we add **gates**, plain Python checks that either fix or reject an intermediate result before it travels further.

Step 1 is where the coverage rule stops being a request. We loop over the three pages and ask for four questions about each one, so an even split across sources is produced by the `for` loop rather than by the model's willingness to count.

In [8]:
class QuestionList(BaseModel):
    questions: list[str] = Field(description="Questions a reader might ask")


prompt_questions = """
Write {n} distinct questions a reader might ask about the documentation page below.

<page title="{title}">
{content}
</page>
""".strip()


def generate_questions(source, n=N_PER_SOURCE):
    """Step 1: questions for ONE page. Coverage is guaranteed by the caller's loop."""
    result = generate_typed(
        prompt_questions.format(
            n=n, title=source["title"], content=source["content"].strip()
        ),
        QuestionList,
    )
    return result.questions[:n]  # gate: never return more than we asked for


questions = generate_questions(page_chunking)
show("Step 1 output for the chunking page",
     "\n".join(f"{i}. {q}" for i, q in enumerate(questions, 1)))


Step 1 output for the chunking page
1. What is the definition of chunking in retrieval systems?
2. What happens when document chunks are either too small or too large?
3. What is the recommended practical default strategy for splitting documents?
4. Why should code blocks and tables never be split?


Step 2 answers one question. Because this call has exactly one job, the word limit is the only instruction competing for attention, and it is followed far more often. When it is not, the gate catches it and spends one small call rewriting the sentence rather than regenerating the page. The retry aims three words under the cap, because a model asked for exactly ten words habitually lands on eleven.

In [9]:
prompt_answer = """
Answer the question in one sentence of at most {max_words} words,
using ONLY the provided content. No preamble, no bullet points.

<question>
{question}
</question>

<provided_content>
{content}
</provided_content>
""".strip()

prompt_shorten = """
Rewrite the sentence below in at most {max_words} words.
Keep the main fact and drop any detail you have to. Count the words.
Return the rewritten sentence only, with no quotes.

<sentence>
{answer}
</sentence>
""".strip()

REPAIRS = {"count": 0}


def answer_question(question, content):
    """Step 2: one answer, with a length gate that repairs instead of hoping."""
    answer = generate(
        prompt_answer.format(
            question=question, max_words=MAX_ANSWER_WORDS, content=content
        )
    ).strip()

    for attempt in range(2):  # gate: at most two repair attempts
        if len(answer.split()) <= MAX_ANSWER_WORDS:
            break
        REPAIRS["count"] += 1
        # Ask for a tighter target on the retry: a model aiming at exactly N
        # words tends to land just over it.
        target = MAX_ANSWER_WORDS if attempt == 0 else MAX_ANSWER_WORDS - 3
        answer = generate(prompt_shorten.format(answer=answer, max_words=target)).strip()

    return answer


answer = answer_question(questions[0], combined_content)
show("Step 2 output", f"Q: {questions[0]}\nA: {answer} ({len(answer.split())} words)")


Step 2 output
Q: What is the definition of chunking in retrieval systems?
A: Splitting a document into pieces is called chunking. (8 words)


Step 3 attributes the answer. The model proposes titles, and the gate keeps only titles that actually exist, so a hallucinated citation is dropped by a set membership test instead of reaching a reader. The page we seeded the question from is always the primary source, because we already know it.

In [10]:
class SourceList(BaseModel):
    sources: list[str] = Field(description="Exact source titles supporting the answer")


prompt_sources = """
You are given a question and an answer written from the documents below.
Return the exact 'Source Title' values whose content supports the answer.
Choose only from these titles: {titles}

<question>
{question}
</question>

<answer>
{answer}
</answer>

<provided_content>
{content}
</provided_content>
""".strip()


def find_sources(question, answer, primary_title, content):
    """Step 3: attribute the answer, keeping only titles that really exist."""
    result = generate_typed(
        prompt_sources.format(
            question=question, answer=answer, content=content, titles=SOURCE_TITLES
        ),
        SourceList,
    )
    verified = [s for s in result.sources if s in SOURCE_TITLES]  # gate
    others = [s for s in verified if s != primary_title]
    return [primary_title] + others


show("Step 3 output", find_sources(questions[0], answer, page_chunking["title"], combined_content))


Step 3 output
['Chunking Documents for Retrieval']


Now the chain itself. It is an ordinary nested loop: three pages, four questions each, two calls per question. Read it once and notice that there is no prompt here at all. The orchestration lives in Python, and each prompt lives inside the one function that owns it.

In [11]:
import time


def sequential_workflow(content):
    faqs = []
    for source in SOURCES:
        for question in generate_questions(source):
            answer = answer_question(question, content)
            sources = find_sources(question, answer, source["title"], content)
            faqs.append(FAQ(question=question, answer=answer, sources=sources))
    return faqs


REPAIRS["count"] = 0
started = time.monotonic()
faqs_chained = sequential_workflow(combined_content)
sequential_seconds = time.monotonic() - started

print(f"sequential chain: {sequential_seconds:.1f} s, length gate fired {REPAIRS['count']} time(s)")
print("check:", check_faqs(faqs_chained))
show("First entry", faqs_chained[0].model_dump_json(indent=2))

sequential chain: 16.9 s, length gate fired 1 time(s)
check: {'returned': 12, 'count_ok': True, 'over_word_limit': 0, 'invented_sources': 0, 'per_source': {'Chunking Documents for Retrieval': 4, 'Embeddings and Vector Search': 4, 'Reranking Retrieved Chunks': 4}, 'all_rules_ok': True}

First entry
{
  "question": "What is the definition of chunking in the context of retrieval systems?",
  "answer": "Splitting a document into pieces is called chunking.",
  "sources": [
    "Chunking Documents for Retrieval"
  ]
}


Every rule now passes by construction rather than by cooperation. The count comes from the loop, the coverage comes from the loop, invented citations are filtered by a set membership test, and the word limit is enforced by the gate. Note the repair counter: the focused answer call still overshoots sometimes, and the difference is that we find out and fix it instead of shipping it.

The price is on the clock. Around thirty calls where the single version made one, and several times the wall time. That is the next problem to fix.

## 6. Parallelization: The Twelve Questions Do Not Know About Each Other

Look at what the chain waits for. Question 7's answer does not depend on question 6's answer, so the loop spends most of its life blocked on network I/O it could have overlapped. **Parallelization** runs independent branches at the same time and combines the results in code.

Two details make this safe. First, `asyncio.to_thread` hands the existing blocking helpers to a worker thread, so the same `answer_question` and `find_sources` functions run unchanged (all three provider SDKs also ship native async clients, for example `gemini_client.aio`, if you would rather go async all the way down). Second, an `asyncio.Semaphore` caps how many calls are in flight, because twelve simultaneous requests is a good way to meet your provider's rate limit and turn a speed-up into a pile of 429s.

In [12]:
import asyncio

MAX_CONCURRENT = 4
limiter = asyncio.Semaphore(MAX_CONCURRENT)


async def process_question(question, primary_title, content):
    """Steps 2 and 3 for one question, in a slot borrowed from the limiter."""
    async with limiter:
        answer = await asyncio.to_thread(answer_question, question, content)
        sources = await asyncio.to_thread(
            find_sources, question, answer, primary_title, content
        )
    return FAQ(question=question, answer=answer, sources=sources)


async def parallel_workflow(content):
    # Step 1 for the three pages at once...
    plans = await asyncio.gather(
        *(asyncio.to_thread(generate_questions, source) for source in SOURCES)
    )
    # ...then steps 2 and 3 for all twelve questions at once.
    jobs = [
        process_question(question, source["title"], content)
        for source, questions in zip(SOURCES, plans)
        for question in questions
    ]
    return await asyncio.gather(*jobs)

Jupyter already runs an event loop, so `asyncio.run()` raises `RuntimeError: This event loop is already running`. Inside a notebook you `await` the coroutine directly in a cell, which is what the next cell does. In a script you would wrap the same call in `asyncio.run(parallel_workflow(...))`.

In [13]:
REPAIRS["count"] = 0
started = time.monotonic()
faqs_parallel = await parallel_workflow(combined_content)
parallel_seconds = time.monotonic() - started

print(f"sequential: {sequential_seconds:5.1f} s")
print(f"parallel:   {parallel_seconds:5.1f} s  "
      f"({sequential_seconds / parallel_seconds:.1f}x faster, "
      f"concurrency capped at {MAX_CONCURRENT})")
print("check:", check_faqs(faqs_parallel))

sequential:  16.9 s
parallel:     5.0 s  (3.4x faster, concurrency capped at 4)
check: {'returned': 12, 'count_ok': True, 'over_word_limit': 0, 'invented_sources': 0, 'per_source': {'Chunking Documents for Retrieval': 4, 'Embeddings and Vector Search': 4, 'Reranking Retrieved Chunks': 4}, 'all_rules_ok': True}


Same twelve entries, same checks, same token bill, a fraction of the wall time. Parallelization buys latency and nothing else: it does not improve quality, and it makes failures messier, because now several calls can fail at once and `asyncio.gather` will raise the first exception while the rest are still running (pass `return_exceptions=True` when you would rather collect them).

## 7. Routing: Classify First, Then Specialize

**Routing** classifies an input and sends it to a handler built for that class. It exists because one prompt that serves every kind of input is a prompt you cannot tune: sharpening it for bug reports makes it worse at billing questions.

You already built one of these in Section 9's "Adding Question Validation and Routing" lesson, where the tutor decided between retrieval, general knowledge, and a polite refusal. Here is the same machinery with the domain removed: a typed classification call, then `if`/`elif`.

The classifier is where an `Enum` earns its place. Typing the field as `Intent` means the model cannot return a category you have no handler for, so the dispatch below has no "unknown label" branch to write.

In [14]:
from enum import Enum


class Intent(str, Enum):
    HOW_TO = "how_to"
    BUG_REPORT = "bug_report"
    PRICING = "pricing"
    OUT_OF_SCOPE = "out_of_scope"


class Classification(BaseModel):
    intent: Intent = Field(description="The category the message belongs to")


prompt_classify = """
Classify the support message into exactly one category.

<categories>
how_to: the user asks how to use a documented feature
bug_report: the user reports something broken, with or without an error message
pricing: the user asks about plans, billing, quotas, or invoices
out_of_scope: anything else, including questions about other products
</categories>

<message>
{message}
</message>
""".strip()


def classify(message):
    return generate_typed(prompt_classify.format(message=message), Classification).intent


for message in ["My uploads over 50 MB fail with error DB-413.",
                "Can you recommend a good laptop for machine learning?"]:
    print(f"{classify(message).value:15s} <- {message}")

bug_report      <- My uploads over 50 MB fail with error DB-413.


out_of_scope    <- Can you recommend a good laptop for machine learning?


Each route then gets a prompt written for one job only. Two things are worth copying from this dispatch. The `out_of_scope` route answers with a fixed string and makes no model call at all, which is the cheapest possible handler. And every route is free to use a different model: classification is a short, easy call that a small cheap model handles well, while the handler that a customer actually reads can be a stronger one.

In [15]:
HANDLER_PROMPTS = {
    Intent.HOW_TO: (
        "You are a documentation assistant. Answer the question in at most three "
        "sentences and name the doc page the user should read next."
        "\n\n<message>\n{message}\n</message>"
    ),
    Intent.BUG_REPORT: (
        "You are a support engineer triaging a bug. In at most three sentences, "
        "restate the failure and ask for the two details you need to reproduce it."
        "\n\n<message>\n{message}\n</message>"
    ),
    Intent.PRICING: (
        "You are a billing assistant. In at most three sentences, acknowledge the "
        "question and say which account details you need to look it up. Never quote "
        "a price you were not given."
        "\n\n<message>\n{message}\n</message>"
    ),
}

REFUSAL = "This desk only covers our own product. Please contact the vendor directly."


def route(message):
    intent = classify(message)
    if intent == Intent.OUT_OF_SCOPE:
        return intent, REFUSAL  # no second call needed
    return intent, generate(HANDLER_PROMPTS[intent].format(message=message)).strip()


inbox = [
    "How do I change the chunk size on an existing collection?",
    "Every upload over 50 MB fails with error DB-413 since yesterday.",
    "Does the team plan include more than 2 million embedded tokens a month?",
    "Can you recommend a good laptop for machine learning?",
]

for message in inbox:
    intent, reply = route(message)
    show(f"[{intent.value}] {message}", reply)


[how_to] How do I change the chunk size on an existing collection?
You cannot change the chunk size on an existing collection because it is immutable after creation. To use a different chunk size, you must create a new collection with your desired settings and migrate your data. 

Read next: **Collections Configuration**



[bug_report] Every upload over 50 MB fails with error DB-413 since yesterday.
Uploads exceeding 50 MB are failing with error DB-413, and this issue began yesterday. To reproduce this, please provide the exact file size/type you are testing with and the API endpoint or region you are targeting.



[pricing] Does the team plan include more than 2 million embedded tokens a month?
I can help you check the details of the team plan regarding embedded token limits. To look this up for you, could you please provide your account email and subscription ID?



[out_of_scope] Can you recommend a good laptop for machine learning?
This desk only covers our own product. Please contact the vendor directly.


The trade-off is that a router adds a call to every request and a new failure mode: a misclassification sends the message to the wrong specialist, and the specialist will answer confidently anyway. Keep the categories few and mutually exclusive, always define the escape category, and treat the classifier as something you evaluate, not something you write once.

## 8. Orchestrator-Workers: When You Cannot Write The Steps In Advance

Routing picks one branch from a list you wrote. Chaining runs steps you fixed in advance. Both assume you know the shape of the work. **Orchestrator-workers** covers the case where you do not: a coordinating call breaks the input into sub-tasks, workers handle them, and a synthesizer merges the results. The number and mix of sub-tasks come from the input [(Anthropic, 2024)](https://www.anthropic.com/engineering/building-effective-agents).

Customer messages are the everyday version of this. One email can contain a billing dispute, a return, and an order lookup in three sentences, and the next one contains two returns.

The orchestrator's whole job is to emit a typed plan. `TaskType` restricts it to work we have a worker for, and the per-task fields are exactly the arguments those workers need.

In [16]:
class TaskType(str, Enum):
    BILLING_INQUIRY = "billing_inquiry"
    PRODUCT_RETURN = "product_return"
    ORDER_STATUS = "order_status"


class Task(BaseModel):
    task_type: TaskType = Field(description="Which worker should handle this task")
    invoice_number: str | None = Field(default=None, description="For billing_inquiry")
    product_name: str | None = Field(default=None, description="For product_return")
    reason_for_return: str | None = Field(default=None, description="For product_return")
    order_id: str | None = Field(default=None, description="For order_status")


class TaskList(BaseModel):
    tasks: list[Task] = Field(description="One task per distinct request in the message")


prompt_orchestrator = """
Break the customer message into a list of tasks, one per distinct request.

Task types and the fields each one needs:
- billing_inquiry: invoice_number
- product_return: product_name, reason_for_return
- order_status: order_id

Leave fields that do not apply empty.

<message>
{message}
</message>
""".strip()


def orchestrate(message):
    return generate_typed(prompt_orchestrator.format(message=message), TaskList).tasks

Now the workers. Only one of the three calls a model: the billing worker has to read the customer's actual complaint out of free text. The other two do what production workers usually do, which is look something up and format the result. A worker is a function that takes a typed task and returns a typed result, and whether an LLM is involved is an implementation detail of that function.

In [17]:
import random


class BillingResult(BaseModel):
    task_type: TaskType = TaskType.BILLING_INQUIRY
    invoice_number: str
    user_concern: str
    case_id: str
    resolution_eta: str


class ReturnResult(BaseModel):
    task_type: TaskType = TaskType.PRODUCT_RETURN
    product_name: str
    reason_for_return: str
    rma_number: str
    shipping_instructions: str


class StatusResult(BaseModel):
    task_type: TaskType = TaskType.ORDER_STATUS
    order_id: str
    current_status: str
    carrier: str
    expected_delivery: str


prompt_billing_concern = """
The customer message below mentions invoice {invoice_number}.
State, in one sentence, what the customer is worried about regarding that invoice.

<message>
{message}
</message>
""".strip()


def billing_worker(task, message):
    """The only worker that needs a model: it reads a complaint out of free text."""
    concern = generate(
        prompt_billing_concern.format(
            invoice_number=task.invoice_number, message=message
        )
    ).strip()
    return BillingResult(
        invoice_number=task.invoice_number,
        user_concern=concern,
        case_id=f"CASE-{random.randint(1000, 9999)}",
        resolution_eta="2 business days",
    )


def return_worker(task, message):
    """Pure Python: an RMA number and fixed instructions, no model involved."""
    rma = f"RMA-{random.randint(10000, 99999)}"
    return ReturnResult(
        product_name=task.product_name,
        reason_for_return=task.reason_for_return or "not stated",
        rma_number=rma,
        shipping_instructions=(
            f"Pack the {task.product_name} in its original box, write {rma} on the "
            "outside, and drop it at any partner pickup point within 14 days."
        ),
    )


# Stands in for the orders table a real worker would query.
ORDER_BOOK = {
    "A-12345": ("Shipped", "SwiftPost", "tomorrow before 18:00"),
    "A-99001": ("Processing", "not assigned yet", "3 to 5 business days"),
}


def status_worker(task, message):
    """Pure Python: a lookup, with a defined answer for the miss case."""
    status, carrier, eta = ORDER_BOOK.get(task.order_id, ("Not found", "n/a", "n/a"))
    return StatusResult(
        order_id=task.order_id,
        current_status=status,
        carrier=carrier,
        expected_delivery=eta,
    )


WORKERS = {
    TaskType.BILLING_INQUIRY: billing_worker,
    TaskType.PRODUCT_RETURN: return_worker,
    TaskType.ORDER_STATUS: status_worker,
}

The synthesizer is the one call that writes prose for a human. It receives the workers' typed results as JSON and is told to cover every fact and invent nothing. Handing it structured data rather than the original message is what keeps the case IDs and RMA numbers correct: they are facts in its input, not things it has to remember.

In [18]:
prompt_synthesizer = """
Write one short reply to a customer, based only on the facts below.
Cover every fact, keep it under 150 words, and do not invent details.
Open with "Hi there," and close with "Best regards, Support".

<facts>
{facts}
</facts>
""".strip()


def synthesize(results):
    facts = "\n\n".join(result.model_dump_json(indent=2) for result in results)
    return generate(prompt_synthesizer.format(facts=facts)).strip()


def handle_message(message):
    tasks = orchestrate(message)
    show("Plan from the orchestrator", "\n".join(t.model_dump_json() for t in tasks))

    results = [WORKERS[task.task_type](task, message) for task in tasks]
    show("Worker results", "\n".join(r.model_dump_json() for r in results))

    return synthesize(results)


customer_message = (
    "Hi, invoice INV-7890 looks about twice what I expected and I would like it "
    "checked. I also need to send back the DeskLamp Pro, it flickers on the lowest "
    "setting. And could you tell me where order A-12345 has got to?"
)

show("Customer message", customer_message)
show("Synthesized reply", handle_message(customer_message))


Customer message
Hi, invoice INV-7890 looks about twice what I expected and I would like it checked. I also need to send back the DeskLamp Pro, it flickers on the lowest setting. And could you tell me where order A-12345 has got to?



Plan from the orchestrator
{"task_type":"billing_inquiry","invoice_number":"INV-7890","product_name":null,"reason_for_return":null,"order_id":null}
{"task_type":"product_return","invoice_number":null,"product_name":"DeskLamp Pro","reason_for_return":"it flickers on the lowest setting","order_id":null}
{"task_type":"order_status","invoice_number":null,"product_name":null,"reason_for_return":null,"order_id":"A-12345"}



Worker results
{"task_type":"billing_inquiry","invoice_number":"INV-7890","user_concern":"The customer is worried that invoice INV-7890 is about twice the amount they expected and wants it checked.","case_id":"CASE-6521","resolution_eta":"2 business days"}
{"task_type":"product_return","product_name":"DeskLamp Pro","reason_for_return":"it flickers on the lowest setting","rma_number":"RMA-22244","shipping_instructions":"Pack the DeskLamp Pro in its original box, write RMA-22244 on the outside, and drop it at any partner pickup point within 14 days."}
{"task_type":"order_status","order_id":"A-12345","current_status":"Shipped","carrier":"SwiftPost","expected_delivery":"tomorrow before 18:00"}



Synthesized reply
Hi there,

Regarding your billing inquiry for invoice INV-7890, where you are worried that it is about twice the amount you expected and want it checked, your case ID is CASE-6521 and the resolution ETA is 2 business days. 

For your product return, the DeskLamp Pro is being returned because it flickers on the lowest setting, and your RMA number is RMA-22244. Please pack the DeskLamp Pro in its original box, write RMA-22244 on the outside, and drop it at any partner pickup point within 14 days.

Additionally, regarding the status of order A-12345, it has shipped with SwiftPost and the expected delivery is tomorrow before 18:00.

Best regards, Support


Change the message and the plan changes with it. Delete the return sentence and the orchestrator emits two tasks, and the same code runs two workers. That adaptivity is the pattern's reason to exist, and it is also its cost: the plan is now model output, so it can be wrong, and every worker you add is a new way for the orchestrator to mis-assign work. Log the plan, cap the number of tasks, and validate each one before dispatch.

In [19]:
short_message = "Where has order A-99001 got to? Also invoice INV-4410 is wrong."
show("Reply to a different message", handle_message(short_message))


Plan from the orchestrator
{"task_type":"order_status","invoice_number":null,"product_name":null,"reason_for_return":null,"order_id":"A-99001"}
{"task_type":"billing_inquiry","invoice_number":"INV-4410","product_name":null,"reason_for_return":null,"order_id":null}



Worker results
{"task_type":"order_status","order_id":"A-99001","current_status":"Processing","carrier":"not assigned yet","expected_delivery":"3 to 5 business days"}
{"task_type":"billing_inquiry","invoice_number":"INV-4410","user_concern":"The customer is worried that invoice INV-4410 is incorrect.","case_id":"CASE-2191","resolution_eta":"2 business days"}



Reply to a different message
Hi there,

Regarding your order A-99001, the current status is processing, a carrier is not assigned yet, and the expected delivery is 3 to 5 business days. 

Regarding your billing inquiry, we understand you are worried that invoice INV-4410 is incorrect. We have opened case ID CASE-2191 for this concern, and the resolution ETA is 2 business days.

Best regards, Support


## 9. What The Whole Notebook Cost

Every helper call was counted, so we can put a number on it. Prices below are the published `gemini-3.5-flash-lite` rates as of August 2026, and you should re-check them against your own provider's page rather than trusting a constant in a notebook.

In [20]:
PRICES_PER_1M = {  # (input, output) USD per 1M tokens, checked August 2026
    "gemini-3.5-flash-lite": (0.30, 2.50),
    "gpt-5.6-luna": (0.20, 1.20),
    "claude-haiku-4-5": (1.00, 5.00),
}

price_in, price_out = PRICES_PER_1M.get(MODELS[PROVIDER], (0.0, 0.0))
cost = (USAGE["input_tokens"] * price_in + USAGE["output_tokens"] * price_out) / 1e6

print(f"model:         {MODELS[PROVIDER]}")
print(f"calls:         {USAGE['calls']}")
print(f"input tokens:  {USAGE['input_tokens']:,}")
print(f"output tokens: {USAGE['output_tokens']:,}")
print(f"estimated cost: ${cost:.4f}")

model:         gemini-3.5-flash-lite
calls:         84
input tokens:  38,036
output tokens: 6,639
estimated cost: $0.0280


## 10. Choosing Between Them

| Pattern | Use it when | It costs you |
| --- | --- | --- |
| Single call | The task is small and every rule can be checked after the fact | No seam to inspect, gate, or repair anything |
| Chaining | Steps have different requirements, or you need to check work in between | More calls, more latency, more code |
| Parallelization | Branches are independent and latency matters | Rate limits, messier error handling |
| Routing | One prompt is being pulled in incompatible directions | An extra call per request, plus misclassification |
| Orchestrator-workers | The sub-tasks depend on the input and cannot be listed in advance | The plan itself can be wrong |

Every pattern here shares one property: **you** wrote the control flow. The model fills in steps, and it never chooses which step runs next. The next lesson removes that constraint and hands the loop to the model, which is what makes something an agent. Before you reach for one, check whether the job you have is a workflow with a fixed shape, because a workflow you can draw is a workflow you can test.